In [1]:
# pip install transformers

In [2]:
# pip install torch torchvision torchaudio -U

In [3]:
# pip install flash-attn --no-build-isolation

In [4]:
# ! pip install flash_attn -U

In [5]:
import torch
torch.cuda.empty_cache()  # Clear cached memory

In [6]:
from torch import bfloat16
import transformers

In [7]:
model_id = "farhananis005/Phi-3-mini-128k-instruct-MoE-v1"

In [8]:
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype=bfloat16,
    device_map='auto',
    attn_implementation='eager'
)


d:\miniconda\envs\ai\Lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/4.70k [00:00<?, ?B/s]

configuration_phi3.py:   0%|          | 0.00/11.2k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-128k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_phi3.py:   0%|          | 0.00/73.2k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-128k-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.
d:\miniconda\envs\ai\Lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/7.64G [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [9]:
model.eval()

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
          (rotary_emb): Phi3LongRoPEScaledRotaryEmbedding()
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLU()
        )
        (input_layernorm): Phi3RMSNorm()
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
        (post_attention_layernorm): Phi3RMSNorm()
      )
    )
    (norm): Phi3RMSNorm()
  )
  (lm_head): Linear(in_features=3072, out

In [10]:
tokenizer = transformers.AutoTokenizer.from_pretrained(model_id)

tokenizer_config.json:   0%|          | 0.00/3.17k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [11]:
# generate_text = transformers.pipeline(
#     model=model, tokenizer=tokenizer,
#     return_full_text=False,  # if using langchain set True
#     task="text-generation",
#     # we pass model parameters here too
#     temperature=0.1,  # 'randomness' of outputs, 0.0 is the min and 1.0 the max
#     top_p=0.15,  # select from top tokens whose probability add up to 15%
#     top_k=0,  # select from top 0 tokens (because zero, relies on top_p)
#     max_new_tokens=2048,  # max number of tokens to generate in the output
#     repetition_penalty=1.1  # if output begins repeating increase
# )

pipe = transformers.pipeline(

    "text-generation",

    model=model,

    tokenizer=tokenizer,

)

In [12]:
generation_args = {

    "max_new_tokens": 1024,

    "return_full_text": False,

    "temperature": 0.5,

    "do_sample": False,

}

In [13]:
sys_msg = """You are a helpful AI assistant, you are an agent capable of using a variety of tools to answer a question. Here are a few of the tools available to you:

- Blog: This tool helps you describe a certain knowledge point and content, and finally write it into Twitter or Facebook style content
- Translate: This is a tool that helps you translate into any language, using plain language as required

To use these tools you must always respond in JSON format containing `"tool_name"` and `"input"` key-value pairs. For example, to answer the question, "Build Muliti Agents with MOE models" you must use the calculator tool like so:

```json

{
    "tool_name": "Blog",
    "input": "Build Muliti Agents with MOE models"
}

```

Or to translate the question "can you introduce yourself in Chinese" you must respond:

```json

{
    "tool_name": "Search",
    "input": "can you introduce yourself in Chinese"
}

```

Remember just output the final result, ouput in JSON format containing `"agentid"`,`"tool_name"` , `"input"` and `"output"`  key-value pairs .:

```json

[


{   "agentid": "step1",
    "tool_name": "Blog",
    "input": "Build Muliti Agents with MOE models",
    "output": "........."
},

{   "agentid": "step2",
    "tool_name": "Search",
    "input": "can you introduce yourself in Chinese",
    "output": "........."
},
{
    "agentid": "final"
    "tool_name": "Result",
    "output": "........."
}
]

```

The users answer is as follows.
"""

In [14]:
def instruction_format(sys_message: str, query: str):
    # note, don't "</s>" to the end
    return f'<|system|> {sys_message} <|end|>\n<|user|> {query} <|end|>\n<|assistant|>'

In [15]:
query ='Write something about Generative AI with MOE , translate it to Vietnamnese'

In [16]:
input_prompt = instruction_format(sys_msg, query)



In [17]:
input_prompt

'<|system|> You are a helpful AI assistant, you are an agent capable of using a variety of tools to answer a question. Here are a few of the tools available to you:\n\n- Blog: This tool helps you describe a certain knowledge point and content, and finally write it into Twitter or Facebook style content\n- Translate: This is a tool that helps you translate into any language, using plain language as required\n\nTo use these tools you must always respond in JSON format containing `"tool_name"` and `"input"` key-value pairs. For example, to answer the question, "Build Muliti Agents with MOE models" you must use the calculator tool like so:\n\n```json\n\n{\n    "tool_name": "Blog",\n    "input": "Build Muliti Agents with MOE models"\n}\n\n```\n\nOr to translate the question "can you introduce yourself in Chinese" you must respond:\n\n```json\n\n{\n    "tool_name": "Search",\n    "input": "can you introduce yourself in Chinese"\n}\n\n```\n\nRemember just output the final result, ouput in JSO

In [18]:
import torch
print(torch.cuda.is_available())  # Should return True
print(torch.cuda.current_device())  # Should return 0 (or another GPU index)

True
0


In [19]:
print(torch.cuda.memory_summary())  # Detailed memory report

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   6020 MiB |   6020 MiB |   6020 MiB |    512 B   |
|       from large pool |   6020 MiB |   6020 MiB |   6020 MiB |      0 B   |
|       from small pool |      0 MiB |      0 MiB |      0 MiB |    512 B   |
|---------------------------------------------------------------------------|
| Active memory         |   6020 MiB |   6020 MiB |   6020 MiB |    512 B   |
|       from large pool |   6020 MiB |   6020 MiB |   6020 MiB |

In [20]:
import torch

torch.cuda.empty_cache() 

import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True "

In [21]:
# res = generate_text(input_prompt)

output = pipe(input_prompt, **generation_args)

d:\miniconda\envs\ai\Lib\site-packages\transformers\generation\configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.5` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
You are not running the flash-attention implementation, expect numerical differences.


In [22]:
output[0]['generated_text']

' ```json\n\n[\n\n    {   "agentid": "step1",\n        "tool_name": "Blog",\n        "input": "Write something about Generative AI with MOE",\n        "output": "........."\n    },\n\n    {   "agentid": "step2",\n        "tool_name": "Translate",\n        "input": "Write something about Generative AI with MOE",\n        "output": "........."\n    },\n\n    {\n        "agentid": "final",\n        "tool_name": "Result",\n        "output": "........."\n    }\n]\n\n``` ```json\n\n[\n\n    {   "agentid": "step1",\n        "tool_name": "Blog",\n        "input": "Write something about Generative AI with MOE",\n        "output": "........."\n    },\n\n    {   "agentid": "step2",\n        "tool_name": "Translate",\n        "input": "Write something about Generative AI with MOE",\n        "output": "........."\n    },\n\n    {\n        "agentid": "final",\n        "tool_name": "Result",\n        "output": "........."\n    }\n]\n\n```\n\nIn this case, the output for each step is represented by ".

In [23]:
import json
import re

def extract_all_json_blocks(text):
    # Tìm tất cả các khối giữa ```json ... ```
    matches = re.findall(r'```json\s*(\[.*?\])\s*```', text, re.DOTALL)

    json_blocks = []
    for json_str in matches:
        try:
            parsed = json.loads(json_str)
            json_blocks.append(parsed)
        except json.JSONDecodeError as e:
            print(f"Lỗi JSON: {e}")
            continue
    return json_blocks

# Giả sử bạn đã có biến `text = output[0]['generated_text']`
text = output[0]['generated_text']
json_arrays = extract_all_json_blocks(text)

for i, block in enumerate(json_arrays):
    print(f"\n JSON Block {i+1}:")
    print(json.dumps(block, ensure_ascii=False, indent=4))



 JSON Block 1:
[
    {
        "agentid": "step1",
        "tool_name": "Blog",
        "input": "Write something about Generative AI with MOE",
        "output": "........."
    },
    {
        "agentid": "step2",
        "tool_name": "Translate",
        "input": "Write something about Generative AI with MOE",
        "output": "........."
    },
    {
        "agentid": "final",
        "tool_name": "Result",
        "output": "........."
    }
]

 JSON Block 2:
[
    {
        "agentid": "step1",
        "tool_name": "Blog",
        "input": "Write something about Generative AI with MOE",
        "output": "........."
    },
    {
        "agentid": "step2",
        "tool_name": "Translate",
        "input": "Write something about Generative AI with MOE",
        "output": "........."
    },
    {
        "agentid": "final",
        "tool_name": "Result",
        "output": "........."
    }
]

 JSON Block 3:
[
    {
        "agentid": "step1",
        "tool_name": "Blog",
       